# Cleaning text in SQL

Practice notebook for the datafromzero SQL lesson **Cleaning text in SQL**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

### Given cell: the hand-typed list

This cell builds `typed_cities`, the eight hand-typed cities of the lesson. It is not part of cafe-sales. Run it once, before the steps below.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE typed_cities (signup_id INTEGER, raw_city VARCHAR);
INSERT INTO typed_cities VALUES
  (1, 'Berkeley'), (2, ' berkeley'), (3, 'BERKELEY '), (4, 'berkeley'),
  (5, 'Oakland, CA'), (6, 'oakland '), (7, 'OAKLAND'), (8, 'Alameda')
""")
print("typed_cities is loaded")

## Worked example

### Step 1. Exact text, on real data

In [ ]:
con.sql("""
SELECT COUNT(*) AS berkeley_customers FROM customers WHERE city = 'berkeley'
""").show()

### Step 2. Change case on the column, not on the table

In [ ]:
con.sql("""
SELECT COUNT(*) AS berkeley_customers FROM customers WHERE LOWER(city) = 'berkeley'
""").show()

### Step 3. Trim the typed list

In [ ]:
con.sql("""
SELECT raw_city, LENGTH(raw_city) AS chars, TRIM(raw_city) AS trimmed, LENGTH(TRIM(raw_city)) AS chars_after
FROM typed_cities
""").show()

### Step 4. Cut out a piece

In [ ]:
con.sql("""
SELECT raw_city, SPLIT_PART(raw_city, ',', 1) AS before_comma
FROM typed_cities
""").show()

### Step 5. Build the key

In [ ]:
con.sql("""
SELECT signup_id, raw_city, LOWER(TRIM(SPLIT_PART(raw_city, ',', 1))) AS city_key
FROM typed_cities
ORDER BY city_key, signup_id
""").show()

### Step 6. Match on the key

In [ ]:
con.sql("""
SELECT COUNT(*) AS berkeley_signups
FROM typed_cities
WHERE LOWER(TRIM(SPLIT_PART(raw_city, ',', 1))) = 'berkeley'
""").show()

### Step 7. Cut and swap on real menu names

In [ ]:
con.sql("""
SELECT product_name,
       UPPER(SUBSTRING(product_name, 1, 3)) AS code,
       REPLACE(product_name, ' ', '_') AS underscored
FROM products
""").show()

### Step 8. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS matched
FROM typed_cities
WHERE LOWER(TRIM(SPLIT_PART(raw_city, ',', 1))) IN ('berkeley', 'oakland', 'alameda')
""").show()

## Exercise 1. A web slug for each tea

The first lines are given: `slug` repeats the name for now. Replace `product_name AS slug` with an expression that uses `REPLACE` and `LOWER`.

In [ ]:
con.sql("""
SELECT product_name, product_name AS slug
FROM products
WHERE category = 'tea'
""").show()

## Exercise 2. Clean a typed sign-up list

Run the given cell first: it builds `typed_signups`, six sign-ups with an email typed by hand (not part of cafe-sales). Before you run anything, write one sentence in the first cell saying what each query will return. Then show each row with `raw_email`, an `email_key` (trimmed, lowercase) and a `domain` (the part after the `@`), and count the sign-ups whose domain is `example.org`.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE typed_signups (signup_id INTEGER, raw_email VARCHAR);
INSERT INTO typed_signups VALUES
  (1, 'Mia@Example.com'), (2, ' mia@example.com'),
  (3, 'NOAH@EXAMPLE.ORG'), (4, 'noah@example.org '),
  (5, 'li@example.net'), (6, 'LI@Example.net ')
""")
print("typed_signups is loaded")

In [ ]:
# My expected result, in words:
#

In [ ]:
# Query 1: the key and the domain of every sign-up

In [ ]:
# Query 2: count the sign-ups on example.org

## Exercise 3. Spot the flaw

A colleague counts the Berkeley sign-ups and says "two", but the list shows four. The query runs without an error. Find the flaw, then write a query that works.

In [ ]:
con.sql("""
SELECT COUNT(*) AS berkeley_signups
FROM typed_cities
WHERE LOWER(raw_city) = 'berkeley'
""").show()

In [ ]:
# Your corrected query: